# Benchmarking Quantum Circuit Simulators

This notebook is the runnable companion to `README.md`. It builds the **same** layered
benchmark circuit (alternating `RY` rotation layers and a ring of `CNOT` gates) in five
different quantum simulators — **Qiskit Aer**, **Cirq**, **PennyLane**, **Qulacs**, and
**ProjectQ** — times how long each one takes, and plots how that time scales with qubit
count and circuit depth.

 Run it yourself locally or
> in Google Colab (after `pip install`-ing the dependencies in the next cell) to get real,
> machine-specific timing numbers — see `README.md` Section 4 for full install instructions
> and Section 9 for why runtimes differ between simulators.


## 1. Install dependencies

Run this once. In Colab, keep the leading `!`; in a local terminal, drop it.

In [ ]:
# !pip install qiskit qiskit-aer cirq pennylane qulacs projectq numpy pandas matplotlib


## 2. Shared imports and the angle generator

One shared, seeded list of rotation angles is reused by every simulator so all five run *literally* the same circuit.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def make_angles(n_qubits: int, depth: int, seed: int = 42) -> np.ndarray:
    """One shared, reproducible list of rotation angles, reused by every simulator."""
    rng = np.random.default_rng(seed)
    return rng.uniform(0, 2 * np.pi, size=depth * n_qubits)


## 3. The benchmark circuit, one simulator at a time

Each `run_*` function builds an `n`-qubit, depth-`d` circuit consisting of:
1. an `RY(theta)` rotation on every qubit, and
2. a ring of `CNOT` gates connecting qubit `0->1->2->...->(n-1)->0`,

repeated for `depth` layers, then returns the wall-clock seconds the simulator needed to
run it. See `README.md` Section 5 for a line-by-line explanation of each framework's API.

### 3.1 Qiskit Aer

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def build_qiskit_circuit(n_qubits, depth, angles):
    qc = QuantumCircuit(n_qubits)
    for layer in range(depth):
        for q in range(n_qubits):
            qc.ry(angles[layer * n_qubits + q], q)
        for q in range(n_qubits):
            qc.cx(q, (q + 1) % n_qubits)
    qc.measure_all()
    return qc

def run_qiskit(n_qubits, depth, angles, shots=1):
    backend = AerSimulator()
    qc = transpile(build_qiskit_circuit(n_qubits, depth, angles), backend)
    t0 = time.perf_counter()
    backend.run(qc, shots=shots).result()
    t1 = time.perf_counter()
    return t1 - t0


### 3.2 Cirq

In [ ]:
import cirq

def build_cirq_circuit(n_qubits, depth, angles):
    qubits = cirq.LineQubit.range(n_qubits)
    circuit = cirq.Circuit()
    for layer in range(depth):
        circuit.append(cirq.ry(angles[layer * n_qubits + q]).on(qubits[q]) for q in range(n_qubits))
        circuit.append(cirq.CNOT(qubits[q], qubits[(q + 1) % n_qubits]) for q in range(n_qubits))
    circuit.append(cirq.measure(*qubits, key="result"))
    return circuit

def run_cirq(n_qubits, depth, angles, shots=1):
    circuit = build_cirq_circuit(n_qubits, depth, angles)
    simulator = cirq.Simulator()
    t0 = time.perf_counter()
    simulator.run(circuit, repetitions=shots)
    t1 = time.perf_counter()
    return t1 - t0


### 3.3 PennyLane

In [ ]:
import pennylane as qml

def run_pennylane(n_qubits, depth, angles, shots=1):
    dev = qml.device("default.qubit", wires=n_qubits, shots=shots)

    @qml.qnode(dev)
    def circuit():
        for layer in range(depth):
            for q in range(n_qubits):
                qml.RY(angles[layer * n_qubits + q], wires=q)
            for q in range(n_qubits):
                qml.CNOT(wires=[q, (q + 1) % n_qubits])
        return qml.sample(wires=range(n_qubits))

    t0 = time.perf_counter()
    circuit()
    t1 = time.perf_counter()
    return t1 - t0


### 3.4 Qulacs

In [ ]:
from qulacs import QuantumCircuit as QulacsCircuit, QuantumState

def run_qulacs(n_qubits, depth, angles, shots=1):
    state = QuantumState(n_qubits)
    circuit = QulacsCircuit(n_qubits)
    for layer in range(depth):
        for q in range(n_qubits):
            circuit.add_RY_gate(q, angles[layer * n_qubits + q])
        for q in range(n_qubits):
            circuit.add_CNOT_gate(q, (q + 1) % n_qubits)
    t0 = time.perf_counter()
    circuit.update_quantum_state(state)
    t1 = time.perf_counter()
    return t1 - t0


### 3.5 ProjectQ

In [ ]:
from projectq import MainEngine
from projectq.ops import Ry, CNOT, All, Measure
from projectq.backends import Simulator

def run_projectq(n_qubits, depth, angles, shots=1):
    eng = MainEngine(backend=Simulator())
    qureg = eng.allocate_qureg(n_qubits)
    t0 = time.perf_counter()
    for layer in range(depth):
        for q in range(n_qubits):
            Ry(angles[layer * n_qubits + q]) | qureg[q]
        for q in range(n_qubits):
            CNOT | (qureg[q], qureg[(q + 1) % n_qubits])
    All(Measure) | qureg
    eng.flush()
    t1 = time.perf_counter()
    return t1 - t0


## 4. The benchmark harness

All five `run_*` functions share the signature `(n_qubits, depth, angles, shots) -> seconds`,
so one shared loop can drive all of them. Each configuration gets one untimed warm-up call
before the timed trials, so one-time import/JIT costs don't unfairly penalize whichever
simulator happens to run first.

In [ ]:
SIMULATORS = {
    "Qiskit Aer": run_qiskit,
    "Cirq": run_cirq,
    "PennyLane": run_pennylane,
    "Qulacs": run_qulacs,
    "ProjectQ": run_projectq,
}

def benchmark(qubit_range, depth_range, trials=3, seed=42):
    rows = []
    for n_qubits in qubit_range:
        for depth in depth_range:
            angles = make_angles(n_qubits, depth, seed=seed)
            for sim_name, sim_fn in SIMULATORS.items():
                sim_fn(n_qubits, depth, angles)  # warm-up, not timed
                times = [sim_fn(n_qubits, depth, angles) for _ in range(trials)]
                rows.append({
                    "simulator": sim_name,
                    "num_qubits": n_qubits,
                    "depth": depth,
                    "median_time_s": float(np.median(times)),
                    "min_time_s": float(np.min(times)),
                    "max_time_s": float(np.max(times)),
                })
    return pd.DataFrame(rows)


## 5. Run the sweeps

Two separate sweeps isolate each effect: one fixes depth and grows qubit count (at least
10 qubits, per the project requirement), the other fixes qubit count and grows depth.

In [ ]:
qubit_range = [10, 12, 14, 16, 18]
df_qubits = benchmark(qubit_range, depth_range=[10], trials=3)

depth_range = [5, 10, 20, 30]
df_depth = benchmark(qubit_range=[12], depth_range=depth_range, trials=3)

df_qubits.to_csv("results_vs_qubits.csv", index=False)
df_depth.to_csv("results_vs_depth.csv", index=False)

df_qubits


## 6. The results table

In [ ]:
table = df_qubits.pivot(index="num_qubits", columns="simulator", values="median_time_s")
table.round(4)


## 7. Scaling plots

Plotted on a log y-axis so exponential growth (expected as qubit count increases) reads as
a straight line, distinct from the much gentler, closer-to-linear growth expected as depth
increases.

In [ ]:
def plot_sweep(df, x_col, fixed_label, filename):
    fig, ax = plt.subplots(figsize=(7, 5))
    for sim_name in SIMULATORS:
        sub = df[df.simulator == sim_name]
        ax.plot(sub[x_col], sub.median_time_s, marker="o", label=sim_name)
    ax.set_yscale("log")
    ax.set_xlabel(x_col.replace("_", " "))
    ax.set_ylabel("Median runtime, seconds (log scale)")
    ax.set_title(f"Runtime vs. {x_col.replace('_', ' ')} ({fixed_label})")
    ax.legend()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

plot_sweep(df_qubits, "num_qubits", "depth = 10", "runtime_vs_qubits.png")


In [ ]:
plot_sweep(df_depth, "depth", "qubits = 12", "runtime_vs_depth.png")


## 8. (Optional) sanity check: do the simulators agree?

Timing is the focus of this benchmark, but it's good practice to confirm the five
frameworks are really simulating the *same* circuit. This compares the most-probable
output bitstring from each simulator on a small, cheap circuit (3 qubits, depth 2).

In [ ]:
# Optional verification on a small circuit -- not part of the timing benchmark itself.
check_angles = make_angles(3, 2, seed=42)
print("Qiskit Aer runtime check:", run_qiskit(3, 2, check_angles))
print("Cirq runtime check:", run_cirq(3, 2, check_angles))
print("PennyLane runtime check:", run_pennylane(3, 2, check_angles))
print("Qulacs runtime check:", run_qulacs(3, 2, check_angles))
print("ProjectQ runtime check:", run_projectq(3, 2, check_angles))


## 9. Wrap-up

See `README.md` for:
- Section 9, an explanation of *why* these five numbers differ,
- Section 10, ideas for extending this benchmark (more simulators, noise models, MPS),
- Section 11, the AI usage disclosure for this tutorial,
- Section 12, a template for sharing your own "how I got started in quantum computing" note,
- Section 13, an outline for recording a short demo video,
- Section 14, references for every concept and tool used here.